<a href="https://colab.research.google.com/github/igorfantucci/Aula-Automatica---GRUPO-5/blob/main/etapa-03-arvores/Aula%2020%20-%20Algoritmo%20de%20Huffman%20-%20code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Demonstrativo: Algoritmo de Huffman na Transmissão de Telemetria SCADA-Core (Grupo 5)

Este notebook demonstra a implementação e simulação do **Algoritmo de Huffman** para compressão sem perdas (*lossless*) de dados de telemetria industrial em um sistema **SCADA** monitorando a **Planta Piloto de Produção de Biodiesel (Transesterificação em Batelada)**.

---

## 1. Contexto Operacional e Mapeamento de Tags ISA-5.1

No ambiente industrial da planta de biodiesel (Grupo 5), o vetor de estado da instrumentação crítica de segurança e processo (Setores 100 e 200) é composto por 5 sensores/transmissores digitais:

1. `PT-201`: Transmissor de Pressão do Reator CSTR R-200 (1 = Sobrepressão crítica $\ge 2.5\text{ bar}$);
2. `TT-201`: Transmissor de Temperatura do Reator R-200 (1 = Sobretemperatura $\ge 65.0^\circ\text{C}$, risco iminente de *flash boiling* do metanol);
3. `AT-100`: Detector de Vapores de Metanol no Setor 100 (1 = Concentração $\ge 20\text{ ppm}$);
4. `LT-201`: Transmissor/Chave de Nível Alto do Reator R-200 (1 = Alarme de transbordamento / nível alto);
5. `ESD-100`: Botoeira de Parada de Emergência Geral do Painel SIS / SIL 3 (1 = Safe state ativado).

O vetor de estado é representado pela palavra de 5 bits: `[PT-201, TT-201, AT-100, LT-201, ESD-100]`.

In [1]:
import heapq
import math
import random
import matplotlib.pyplot as plt
import pandas as pd

# Configuração estática de visualização
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid' if 'seaborn-v0_8-darkgrid' in plt.style.available else 'default')

## 2. Modelo Probabilístico das Mensagens da Planta

Devido ao regime operacional contínuo e estável do reator de transesterificação em batelada, o estado de operação normal estável (`00000`) ocorre na imensa maioria das leituras de telemetria ($90.0\%$ das amostras). Desvios de nível, sobrepressão, sobretemperatura, vazamento de metanol ou parada de emergência ocorrem com frequências decrescentes.

In [2]:
# Tabela de probabilidades de cada estado lógico [PT-201, TT-201, AT-100, LT-201, ESD-100]
state_probabilities = {
    '00000': 0.900,  # Operação Nominal Estável (Setores 100 a 400)
    '00010': 0.050,  # Nível Alto Transitório na Carga do Reator (LT-201)
    '10000': 0.025,  # Alarme de Sobrepressão no Reator (PT-201 >= 2.5 bar)
    '01000': 0.015,  # Alarme de Sobretemperatura (TT-201 >= 65°C - Flash Boiling)
    '00100': 0.009,  # Detector de Vapores de Metanol Setor 100 (AT-100 >= 20 ppm)
    '00001': 0.001   # Botoeira de Parada de Emergência SIS Acionada (ESD-100)
}

total_p = sum(state_probabilities.values())
print(f"Soma total das probabilidades: {total_p:.3f}")
assert math.isclose(total_p, 1.0), "As probabilidades devem somar 1.0"

Soma total das probabilidades: 1.000


## 3. Construção da Árvore de Huffman e Geração dos Códigos de Prefixo

Implementamos a estrutura de nós da árvore binária e utilizamos uma fila de prioridade (*Min-Heap*) para construir a árvore ótima de prefixo livre (*Prefix-Free Codes*).

In [3]:
class HuffmanNode:
    def __init__(self, symbol=None, freq=0, left=None, right=None):
        self.symbol = symbol
        self.freq = freq
        self.left = left
        self.right = right

    def __lt__(self, other):
        return self.freq < other.freq

def build_huffman_tree(probabilities):
    heap = [HuffmanNode(symbol=sym, freq=p) for sym, p in probabilities.items()]
    heapq.heapify(heap)

    while len(heap) > 1:
        left = heapq.heappop(heap)
        right = heapq.heappop(heap)
        # Convenção: bit 0 para maior probabilidade (left) e 1 para menor (right)
        if left.freq < right.freq:
            left, right = right, left
        merged = HuffmanNode(freq=left.freq + right.freq, left=left, right=right)
        heapq.heappush(heap, merged)

    return heap[0]

def generate_codes(root, current_code="", code_dict=None):
    if code_dict is None:
        code_dict = {}
    if root is not None:
        if root.symbol is not None:
            code_dict[root.symbol] = current_code
        generate_codes(root.left, current_code + "0", code_dict)
        generate_codes(root.right, current_code + "1", code_dict)
    return code_dict

# Construir árvore e gerar dicionário de códigos
tree_root = build_huffman_tree(state_probabilities)
huffman_codes = generate_codes(tree_root)

print("\n--- Códigos de Huffman Gerados para a Planta de Biodiesel ---")
for sym, code in huffman_codes.items():
    print(f"Estado: {sym} | Probabilidade: {state_probabilities[sym]:6.3f} | Código Huffman: {code:>5} ({len(code)} bits)")


--- Códigos de Huffman Gerados para a Planta de Biodiesel ---
Estado: 00000 | Probabilidade:  0.900 | Código Huffman:     0 (1 bits)
Estado: 00010 | Probabilidade:  0.050 | Código Huffman:    10 (2 bits)
Estado: 10000 | Probabilidade:  0.025 | Código Huffman:   110 (3 bits)
Estado: 01000 | Probabilidade:  0.015 | Código Huffman:  1110 (4 bits)
Estado: 00100 | Probabilidade:  0.009 | Código Huffman: 11110 (5 bits)
Estado: 00001 | Probabilidade:  0.001 | Código Huffman: 11111 (5 bits)


## 4. Simulação de Transmissão SCADA em Tempo Real

Simulamos uma transmissão contínua de **10.000 amostras** de telemetria enviadas pelos transmissores do CLP de campo para o Servidor SCADA central através de um canal serial de telemetria.

In [4]:
# Gerar série temporal com 10.000 amostras de amostragem SCADA
NUM_SAMPLES = 10000
symbols = list(state_probabilities.keys())
weights = list(state_probabilities.values())

random.seed(42)
telemetry_stream = random.choices(symbols, weights=weights, k=NUM_SAMPLES)

# 1. Codificação bruta em tamanho fixo (5 bits por amostra)
raw_bitstream = "".join(telemetry_stream)
raw_size_bits = len(raw_bitstream)

# 2. Codificação comprimida via Huffman
compressed_bitstream = "".join([huffman_codes[sym] for sym in telemetry_stream])
compressed_size_bits = len(compressed_bitstream)

print(f"Amostras Processadas:           {NUM_SAMPLES:,}")
print(f"Tamanho Bruto (Formato Fixo):   {raw_size_bits:,} bits ({raw_size_bits/8/1024:.2f} KB)")
print(f"Tamanho Comprimido (Huffman):   {compressed_size_bits:,} bits ({compressed_size_bits/8/1024:.2f} KB)")
reduction_pct = 100 * (1 - compressed_size_bits / raw_size_bits)
print(f"Redução total de tráfego:       {reduction_pct:.2f}%")

Amostras Processadas:           10,000
Tamanho Bruto (Formato Fixo):   50,000 bits (6.10 KB)
Tamanho Comprimido (Huffman):   11,878 bits (1.45 KB)
Redução total de tráfego:       76.24%


## 5. Algoritmo de Decodificação e Validação Sem Perdas (*Lossless*)

No receptor SCADA, o fluxo binário recebido é percorrido bit a bit sobre a árvore de Huffman para reconstruir a sequência original de telemetria, assegurando fidelidade integral de dados para os laços de segurança SIS.

In [5]:
def decode_huffman(bitstream, root):
    decoded_symbols = []
    current = root
    for bit in bitstream:
        if bit == '0':
            current = current.left
        else:
            current = current.right

        if current.symbol is not None:
            decoded_symbols.append(current.symbol)
            current = root
    return decoded_symbols

# Executar decodificação no servidor SCADA
decoded_telemetry = decode_huffman(compressed_bitstream, tree_root)

# Validação formal de igualdade exata (Lossless Test)
is_equal = (telemetry_stream == decoded_telemetry)
assert is_equal, "Erro crítico: os dados recuperados não conferem com o original!"
print(f"Validação de Integridade (Lossless Verification): {'✅ SUCESSO! Recuperação 100% fiel sem perdas.' if is_equal else '❌ ERRO na integridade!'}")

Validação de Integridade (Lossless Verification): ✅ SUCESSO! Recuperação 100% fiel sem perdas.


## 6. Análise Teórica e Gráficos de Desempenho

Comparamos os resultados experimentais com o limite termodinâmico/informacional da **Entropia de Shannon** $H(S)$.

In [6]:
# Entropia de Shannon H(S)
entropy = -sum(p * math.log2(p) for p in state_probabilities.values())

# Comprimento Médio Teórico L
avg_length = sum(state_probabilities[sym] * len(huffman_codes[sym]) for sym in state_probabilities)

print(f"Entropia de Shannon H(S):          {entropy:.4f} bits/símbolo")
print(f"Comprimento Médio da Palavra L:    {avg_length:.4f} bits/símbolo")
print(f"Comprimento do Formato Fixo:       5.0000 bits/símbolo")
print(f"Eficiência Teórica de Compressão:  {(1 - avg_length/5)*100:.2f}%")

# Gráficos comparativos de desempenho
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Visualização 1: Comparação de Volume Transmitido (KBytes)
categories = ['Sem Compressão\n(Formato Fixo 5 bits)', 'Com Compressão\n(Huffman SCADA)']
values_kb = [raw_size_bits / 8 / 1024, compressed_size_bits / 8 / 1024]
colors = ['#e74c3c', '#2ecc71']

bars = ax1.bar(categories, values_kb, color=colors, width=0.45)
ax1.set_ylabel('Volume de Dados (KBytes)', fontsize=11)
ax1.set_title('Volume Total Transmitido em 10.000 Amostras SCADA', fontsize=12, fontweight='bold')
for bar in bars:
    yval = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, yval + 0.1, f'{yval:.2f} KB', ha='center', va='bottom', fontweight='bold')

# Visualização 2: Comprimento dos Códigos vs Frequência Operacional
states = list(state_probabilities.keys())
probs = [state_probabilities[s]*100 for s in states]
code_lens = [len(huffman_codes[s]) for s in states]

ax2.bar(states, code_lens, color='#3498db', alpha=0.85, label='Comprimento do Código (bits)')
ax2_twin = ax2.twinx()
ax2_twin.plot(states, probs, color='#f39c12', marker='o', linewidth=2.5, label='Probabilidade de Ocorrência (%)')

ax2.set_xlabel('Vetor de Estado [PT-201, TT-201, AT-100, LT-201, ESD-100]', fontsize=10)
ax2.set_ylabel('Comprimento do Código (bits)', fontsize=11, color='#2980b9')
ax2_twin.set_ylabel('Frequência de Ocorrência (%)', fontsize=11, color='#d35400')
ax2.set_title('Relação Frequência vs Tamanho do Código de Huffman', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

Entropia de Shannon H(S):          0.6480 bits/símbolo
Comprimento Médio da Palavra L:    1.1850 bits/símbolo
Comprimento do Formato Fixo:       5.0000 bits/símbolo
Eficiência Teórica de Compressão:  76.30%
